<a href="https://colab.research.google.com/github/haydenhollar/ai301-vol-gig/blob/main/notebooks/checkpoint1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vol Gig matching agent
**AI 301 Team 6 · Hayden Hollar and Jefferson Vicente**

Progress Note

Completed so far:

We used Claude to generate the CSV files for the 100 student profiles and 20 job postings. We have 20 test situations with answer keys. Two of them are ones our data cannot answer. Both baselines are scored. BM25 has a recall of 0.861 and the fixed script scored 0.30. Each of the four tools has been tested on its own.


Proposal feedback:

Dr. Kidane recommended we reduce our tools from our original seven down to four and keeping our CSVs instead of changing them to JSON format.

Design:

Our proposal tests one combined vector per student against two vectors. One would cover the dropdown options from signup. The other would cover the paragraph the student writes. We have not run that test yet.

Who worked on what:

Hayden: Came up with the project idea. Chose which four tools to keep and how the posting form works. Set up the GitHub repository and added Jefferson. Gathered the class labs we pull code from. Directed Claude's work and reviewed what it produced. Checked this notebook cell by cell in Colab, added the zips check, ran everything and wrote this note.

Jefferson: Helped write the proposal. Reviewed and approved the project plan with Hayden. Most of his work comes in Sprint 2, where he owns the matching code and the mock student replies.

Claude: Generated the data files and the test set. Wrote the shared code in src/volgig.py, the baseline cells and the setup for this notebook. Found the class lab code we used, which is the BM25 keyword search from Module 8 and the duplicates check from the midterm lab.

All design choices were made by us.

---

## Part 0 · Setup

Part 0

In [1]:
# Setup: get the latest repo files and load the data. Run this first, every session.
import os, subprocess, sys
REPO_URL = "https://github.com/haydenhollar/ai301-vol-gig.git"
REPO = "/content/repo"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rank_bm25"], check=True)

sys.path.insert(0, f"{REPO}/src")
import importlib, volgig as vg
importlib.reload(vg)
import pandas as pd

data = vg.load_data(f"{REPO}/data")
RESULTS = f"{REPO}/results"
os.makedirs(RESULTS, exist_ok=True)
for k in ["students", "postings", "test_set", "planted", "zips"]:
    print(f"{k}: {data[k].shape}")

students: (100, 11)
postings: (20, 13)
test_set: (20, 14)
planted: (7, 6)
zips: (17, 3)


Data

Our students, postings, test situations and planted cases are all synthetic based on real students and job postings from other job marketplaces.

In order to get accurate ZIP code locations and distances we used one outside source which requires acknowledgement. GeoNames provided postal code data thorugh the pgeocode Python package. Their data is shared under a Creative Commons Atrtribution license.

Size

100 students, 20 job postings, 20 test situations and 7 planted cases.
17 ZIP codes from GeoNames.

In [2]:
# Duplicates: the same ID twice, or the same posting text twice
students = data["students"]
postings = data["postings"]
print("repeated student IDs:", students["student_id"].duplicated().sum())
print("repeated posting IDs:", postings["posting_id"].duplicated().sum())
print("repeated posting descriptions:", postings["description"].duplicated().sum())

# remove any repeats, keeping the first copy; ~ flips True and False, so this keeps the rows that are not repeats
data["students"] = students[~students["student_id"].duplicated()]
data["postings"] = postings[~postings["posting_id"].duplicated() & ~postings["description"].duplicated()]
print("students:", len(students), "->", len(data["students"]), "| postings:", len(postings), "->", len(data["postings"]))

repeated student IDs: 0
repeated posting IDs: 0
repeated posting descriptions: 0
students: 100 -> 100 | postings: 20 -> 20


---

## Part 1 · A look at the data

In [3]:
s = data["students"].loc["S12"]
p = data["postings"].loc["J07"]

print(s[["student_id", "major", "year", "availability", "home_zip", "travel_radius_miles", "has_car", "tools"]].to_dict())
print("search reads:", vg.student_text(s))
print()
print(p[["posting_id", "kind", "open_or_private", "category", "time_slots", "zip"]].to_dict())
print("search reads:", vg.posting_text(p))

{'student_id': 'S12', 'major': 'Mechanical Engineering', 'year': 'Junior', 'availability': ['Sat morning', 'Sun afternoon'], 'home_zip': '37919', 'travel_radius_miles': 10.0, 'has_car': 'yes', 'tools': ['ladder']}
search reads: lawn mowing, moving help. Grew up on a farm. Have run a chainsaw since I was twelve.

{'posting_id': 'J07', 'kind': 'odd job', 'open_or_private': 'open', 'category': 'tree work', 'time_slots': ['Sat morning', 'Sat afternoon', 'Sun morning', 'Sun afternoon'], 'zip': '37920'}
search reads: Need some tree limbs cut down this weekend. I have a ladder. $25 an hour. 37920 tree work


In [4]:
passed = vg.hard_filter("J07", data)
print("students who pass the hard filters for J07:", len(passed))

reasons = {}
for sid, student in data["students"].iterrows():
    reason = vg.hard_filter_reason(student, p, data["zips"])
    reasons[reason] = reasons.get(reason, 0) + 1
print(reasons)

students who pass the hard filters for J07: 46
{'distance': 12, 'availability': 42, 'pass': 46}


In [5]:
display(data["test_set"][["situation_id", "posting_id", "poster_message", "expected_tools", "expected_args", "answerable"]])
display(data["planted"][["posting_id", "case", "should_match", "close_but_wrong"]])

,situation_id,posting_id,event_type,poster_message,expected_tools,difficulty
0,T01,J01,new_posting,A new posting was just submitted.,send_match_list; send_invite,easy
1,T02,J03,new_posting,A new posting was just submitted.,send_match_list; send_invite,easy
2,T03,J07,new_posting,A new posting was just submitted.,send_match_list; send_invite,easy
3,T04,J15,new_posting,A new posting was just submitted.,send_match_list; send_invite,medium
4,T05,J13,new_posting,A new posting was just submitted.,send_match_list; send_invite,medium
5,T06,J20,new_posting,A new posting was just submitted.,send_match_list; send_invite,medium
6,T07,J10,new_posting,A new posting was just submitted.,send_match_list,medium
7,T08,J10,poster_message,Please invite S07 and S31.,send_invite,hard


,posting_id,case,should_match,close_but_wrong
0,J01,Many matches: mow lawn Saturday morning 37919,"S03, S05, S09, S12, S18, S33, S36, S49, S51, S...","S22, S27, S44, S87, S100"
1,J03,Truck needed: haul furniture Saturday 37920,"S09, S36, S51, S54, S99","S76, S97"
2,J07,Tree limbs this weekend 37920,"S12, S33, S58","S41, S73, S79"
3,J13,Grandmother's letters from Osaka,"S28, S53","S47, S71, S74"
4,J15,Unload a truck Sunday at 6 AM 37922,"S19, S97","S12, S36, S62"
5,J10,Private job: algebra tutoring,Invite S07 and S31 only,"Must not be notified: S02, S38, S52, S72, S78,..."
6,J18,Licensed electrician,Nobody,"S35, S54, S75"


---

## Part 2 · The two baselines

In [5]:
import re
from rank_bm25 import BM25Okapi

STOP = set("""a an and are as at be been but by can do for from had has have i i'm if in into is it its
just me my no not of on or our so that the their them there they this to up was we were will with you your
about after all also am any been both each every get got how more most need needs one only over some than
too very what when which who would looking""".split())

def tokens(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]

students = data["students"]
bm25 = BM25Okapi([tokens(vg.student_text(s)) for _, s in students.iterrows()])
row_of = {sid: i for i, sid in enumerate(students.index)}

def bm25_matches(posting_id):
    p = data["postings"].loc[posting_id]
    scores = bm25.get_scores(tokens(vg.posting_text(p)))
    passed = vg.hard_filter(posting_id, data)
    ranked = sorted(((scores[row_of[s]], s) for s in passed), reverse=True)
    return [s for score, s in ranked if score > 0]

bm25_rows = [vg.matching_scores(bm25_matches(r["posting_id"]), r) for _, r in data["planted"].iterrows()]
bm25_table = pd.DataFrame(bm25_rows)
bm25_table.to_csv(f"{RESULTS}/baseline_bm25_matching.csv", index=False)
bm25_table

,posting_id,should_match,found,recall,missed,near_misses_leaked,list_size
0,J01,12,12,1.000000,,,12
1,J03,5,5,1.000000,,,5
2,J07,3,2,0.666667,S12,S73,6
3,J13,2,1,0.500000,S53,S47 S71,8
4,J15,2,2,1.000000,,,2
5,J10,2,2,1.000000,,,4
6,J18,0,0,NaN,,,0


In [6]:
def fixed_script(board, posting_id, matches):
    board.send_match_list(posting_id, matches)
    board.send_invite(posting_id, matches)
    return ""  # the fixed script never writes a reply

fixed_rows, fixed_logs = [], []
for _, sit in data["test_set"].iterrows():
    matches = bm25_matches(sit["posting_id"])
    board = vg.JobBoard(sit, matches, data)
    reply = fixed_script(board, sit["posting_id"], matches)
    fixed_rows.append(vg.score_situation(sit, board.log, reply, matches))
    for c in board.log:
        fixed_logs.append({"situation_id": sit["situation_id"], **c})

fixed_table = pd.DataFrame(fixed_rows)
fixed_table.to_csv(f"{RESULTS}/baseline_fixed_script.csv", index=False)
pd.DataFrame(fixed_logs).to_csv(f"{RESULTS}/baseline_fixed_script_log.csv", index=False)
fixed_table

,situation_id,difficulty,answerable,expected,called,action_correct,task_success,contacted_forbidden
0,T01,easy,yes,send_match_list; send_invite,send_match_list; send_invite,True,True,
1,T02,easy,yes,send_match_list; send_invite,send_match_list; send_invite,True,True,
2,T03,easy,yes,send_match_list; send_invite,send_match_list; send_invite,True,True,
3,T04,medium,yes,send_match_list; send_invite,send_match_list; send_invite,True,True,
4,T05,medium,yes,send_match_list; send_invite,send_match_list; send_invite,True,True,
5,T06,medium,yes,send_match_list; send_invite,send_match_list; send_invite,True,True,
6,T07,medium,yes,send_match_list,send_match_list; send_invite,False,False,S07 S31 S52 S94
7,T08,hard,yes,send_invite,send_match_list; send_invite,False,False,S52 S94
8,T09,medium,yes,send_invite,send_match_list; send_invite,False,False,S94
9,T10,medium,yes,send_match_list,send_match_list; send_invite,False,False,


In [7]:
m = bm25_table.dropna(subset=["recall"])
summary = pd.DataFrame([
    {"baseline": "BM25 keyword search", "measure": "Matching recall, mean over planted postings",
     "score": round(m["recall"].mean(), 3), "n": len(m)},
    {"baseline": "BM25 keyword search", "measure": "Near-misses that leaked onto a list",
     "score": int(sum(len(x.split()) for x in bm25_table["near_misses_leaked"])), "n": len(bm25_table)},
    {"baseline": "Fixed script", "measure": "Action correct (share of 20 situations)",
     "score": round(fixed_table["action_correct"].mean(), 3), "n": len(fixed_table)},
    {"baseline": "Fixed script", "measure": "Task success (share of 20 situations)",
     "score": round(fixed_table["task_success"].mean(), 3), "n": len(fixed_table)},
    {"baseline": "Fixed script", "measure": "Situations where a forbidden student was contacted",
     "score": int((fixed_table["contacted_forbidden"] != "").sum()), "n": len(fixed_table)},
])
summary.to_csv(f"{RESULTS}/baseline_scores.csv", index=False)
display(summary)
print("Fixed script task success by difficulty:")
display(fixed_table.groupby("difficulty")["task_success"].agg(["mean", "count"]))

,baseline,measure,score,n
0,BM25 keyword search,"Matching recall, mean over planted postings",0.861,6
1,BM25 keyword search,Near-misses that leaked onto a list,3.000,7
2,Fixed script,Action correct (share of 20 situations),0.300,20
3,Fixed script,Task success (share of 20 situations),0.300,20
4,Fixed script,Situations where a forbidden student was conta...,9.000,20


Fixed script task success by difficulty:


,mean,count
difficulty,,
easy,0.428571,7
hard,0.000000,6
medium,0.428571,7


In [8]:
# Core component: each of the four tools called on its own, with no model
sit = data["test_set"].iloc[12]                  # T13 on job J07: S12 and S33 accepted, S58 has not answered
matches = bm25_matches(sit["posting_id"])
board = vg.JobBoard(sit, matches, data)

print("send_match_list:", board.send_match_list(sit["posting_id"], matches))
print("send_invite:", board.send_invite(sit["posting_id"], matches))
print("get_status:", board.get_status(sit["posting_id"]))
print("book_student, S58 has not accepted:", board.book_student(sit["posting_id"], "S58"))
print("book_student, S12 accepted:", board.book_student(sit["posting_id"], "S12"))

# the private-job rule on J10: only the students the poster named can be invited
private = data["test_set"].iloc[7]               # T08: "Please invite S07 and S31."
private_matches = bm25_matches(private["posting_id"])
private_board = vg.JobBoard(private, private_matches, data)
print("send_invite to every match on a private job:", private_board.send_invite(private["posting_id"], private_matches))
print("send_invite to the two the poster named:", private_board.send_invite(private["posting_id"], ["S07", "S31"]))

print()
for call in board.log + private_board.log:
    print(call)

send_match_list: Poster received 6 students.
send_invite: Invited ['S73', 'S70', 'S58', 'S33', 'S22', 'S03'].
get_status: {'accepted': ['S12', 'S33'], 'declined': [], 'no answer yet': ['S58', 'S73', 'S70', 'S22', 'S03']}
book_student, S58 has not accepted: Refused: S58 has not accepted.
book_student, S12 accepted: Booked S12. 6 other invited students told the job is filled.
send_invite to every match on a private job: Refused: ['S52', 'S94'] were not picked by the poster.
send_invite to the two the poster named: Invited ['S07', 'S31'].

{'tool': 'send_match_list', 'students': ['S73', 'S70', 'S58', 'S33', 'S22', 'S03'], 'allowed': True, 'result': 'Poster received 6 students.'}
{'tool': 'send_invite', 'students': ['S73', 'S70', 'S58', 'S33', 'S22', 'S03'], 'allowed': True, 'result': "Invited ['S73', 'S70', 'S58', 'S33', 'S22', 'S03']."}
{'tool': 'get_status', 'students': [], 'allowed': True, 'result': {'accepted': ['S12', 'S33'], 'declined': [], 'no answer yet': ['S58', 'S73', 'S70', 'S2